# Run simulations for viscous eddies

Part of the simulations done for the paper with Erbach

### First load some references

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.PrintingNip;
Init();

### Init Database etc.

In [ ]:
string ProjectName = "ViscousEddies";

In [3]:
BoSSSshell.WorkflowMgm.Init(ProjectName);

In [4]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [5]:
static var myBatch = BoSSSshell.GetDefaultQueue();
myBatch // print queue information

In [6]:
static var myDb = BoSSSshell.WorkflowMgm.DefaultDatabase;

In [7]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

### Setup Simulations

The coordiante transformation from (x,y) to (&xi;,&eta;) can be represented by:  
Where represents c is the y coordinate of the focal points of the coordinate system.  

$ x+iy = -c\,\cot(0.5(\xi+i\eta)) $

or

$ x = \frac{-c\,\sin(\xi)}{\cosh(\eta) - \cos(\xi)} $  
$ y = \frac{c\,\sinh(\eta)}{\cosh(\eta) - \cos(\xi)} $

Following a transformation $ (x,y) = T(\xi,\eta) $ is defined.  
To this end the position of the focal point has to be determined in dependance of cylinder radius $ R $ and nip width $ \epsilon $.

$ c = R\left| \sinh\left(\cosh^{-1}\left( \frac{\epsilon}{R} + 1\right)\right) \right|$

The transformation also stores the minimum and maximum value for $ \xi $ and $ \eta $ used later in the grid generation.  
These are calculated as:

$ \eta = 0.5\,\ln\left(\frac{(y+c)^2+x^2}{(y-c)^2+x^2}\right) $  
$ \xi = \pi + \tan^{-1}\left(\frac{x}{y+c}\right) - \tan^{-1}\left(\frac{x}{y-c}\right) $


The different solution branches of the Stream function are, in parenthesis the real value of the parameter alpha for the first two modes and $h/R = 0.01$:  
Odd, asymmetric ($\alpha_0 = 8.0152 + 14.896i, \alpha_1 = 11.028 + 37.902i$):  
$\Psi_{o,a}(\xi,\eta;\alpha_n)  =  A E\ \frac{c}{\cosh(\eta)-\cos(\xi)}\left[\cos(\alpha_n\xi)- \cot(\pi \alpha_n)\sin(\alpha_n\xi)\right] \left[\cosh((\alpha_n+1)\eta)-\frac{\cosh((\alpha_n+1)\eta_0)}{\cosh((\alpha_n-1)\eta_0)}\cosh((\alpha_n-1)\eta)\right].$

Even, asymmetric ($\alpha_0 = 8.0152 + 14.896i, \alpha_1 = 11.028 + 37.902i$):  
$\Psi_{e,a}(\xi,\eta;\alpha_n)  = A E\ \frac{c}{\cosh(\eta)-\cos(\xi)}\left[\cos(\alpha_n\xi)+\tan(\pi \alpha_n)\sin(\alpha_n\xi)\right] \left[\cosh((\alpha_n+1)\eta)-\frac{\cosh((\alpha_n+1)\eta_0)}{\cosh((\alpha_n-1)\eta_0)}\cosh((\alpha_n-1)\eta)\right]$

Odd, symmetric ($\alpha_0 = 9.8456 + 26.525i, \alpha_1 = 11.909 + 49.181i$):  
$\Psi_{o,s}(\xi,\eta;\alpha_n)  = B E\ \frac{c}{\cosh(\eta)-\cos(\xi)}\left[\cos(\alpha_n\xi)- \cot(\pi \alpha_n)\sin(\alpha_n\xi)\right] \left[\sinh((\alpha_n+1)\eta)-\frac{\sinh((\alpha_n+1)\eta_0)}{\sinh((\alpha_n-1)\eta_0)}\sinh((\alpha_n-1)\eta)\right]$

Even, symmetric ($\alpha_0 = 9.8456 + 26.525i, \alpha_1 = 11.909 + 49.181i$):  
$\Psi_{e,s}(\xi,\eta;\alpha_n)  = B E\ \frac{c}{\cosh(\eta)-\cos(\xi)}\left[\cos(\alpha_n\xi)+\tan(\pi \alpha_n)\sin(\alpha_n\xi)\right] \left[\sinh((\alpha_n+1)\eta)-\frac{\sinh((\alpha_n+1)\eta_0)}{\sinh((\alpha_n-1)\eta_0)}\sinh((\alpha_n-1)\eta)\right]$

Numerically the following calculations are more robust w.r.t. to finite precision arithmetic:  
$\left[\cos(\alpha_n\xi)-\cot(\pi \alpha_n)\sin(\alpha_n\xi)\right] = -\frac{\sin(\alpha_n(\xi-\pi))}{\sin(\alpha_n\pi)}$  
$\left[\cos(\alpha_n\xi)+\tan(\pi \alpha_n)\sin(\alpha_n\xi)\right] = \frac{\cos(\alpha_n(\xi-\pi))}{\cos(\alpha_n\pi)}$  

In [8]:
using MathNet.Numerics;
using System.Numerics;

In [9]:
public static class BoundaryValueFactory4Eddies {
        public static string GetPrefixCode(int mode, bool even, bool symmetric, double _delta, double Radius = 0.1) {
            using (var stw = new System.IO.StringWriter()) {  
                stw.WriteLine("using MathNet.Numerics;");
                stw.WriteLine("using System.Numerics;");
                stw.WriteLine("static class BoundaryValues {");
                stw.WriteLine("  static public double Streamfunction(double _xi, double _eta) {");
                stw.WriteLine("    Complex xi = new Complex(_xi, 0.0);");
                stw.WriteLine("    Complex eta = new Complex(_eta, 0.0);");
                stw.WriteLine("    Complex alpha;");
                stw.WriteLine("    Complex PsiXi;");
                stw.WriteLine("    Complex PsiEta;");
                stw.WriteLine("    if(symmetric){");
                stw.WriteLine("        alpha = Alpha[1, mode];");
                stw.WriteLine("        PsiEta = System.Numerics.Complex.Sinh((alpha+1.0)*eta)-System.Numerics.Complex.Sinh((alpha+1.0)*eta0)/System.Numerics.Complex.Sinh((alpha-1.0)*eta0)*System.Numerics.Complex.Sinh((alpha-1.0)*eta);");
                stw.WriteLine("    } else {");
                stw.WriteLine("        alpha = Alpha[0, mode];");
                stw.WriteLine("        PsiEta = System.Numerics.Complex.Cosh((alpha+1.0)*eta)-System.Numerics.Complex.Cosh((alpha+1.0)*eta0)/System.Numerics.Complex.Cosh((alpha-1.0)*eta0)*System.Numerics.Complex.Cosh((alpha-1.0)*eta);");
                stw.WriteLine("    }");
                stw.WriteLine("    if(even){");
                stw.WriteLine("        PsiXi = System.Numerics.Complex.Cos(alpha*(xi-Math.PI))/System.Numerics.Complex.Cos(alpha*Math.PI);");
                stw.WriteLine("    } else {");
                stw.WriteLine("        PsiXi = -System.Numerics.Complex.Sin(alpha*(xi-Math.PI))/System.Numerics.Complex.Sin(alpha*Math.PI);");
                stw.WriteLine("    }");
                stw.WriteLine("    Complex A = new Complex(Math.Exp(alpha.Imaginary ), 0.0);");
                stw.WriteLine("    Complex scale = c/(System.Numerics.Complex.Cosh(eta)-System.Numerics.Complex.Cos(xi));");
                stw.WriteLine("    Complex Psi = A*scale*PsiXi*PsiEta;");
                stw.WriteLine("    return Psi.Real;");
                stw.WriteLine("  }");
                stw.WriteLine("  static public Complex[,] Alpha = {{new Complex(0.0, 0.0), new Complex(8.0152, 14.896), new Complex(11.028, 37.902)},{ new Complex(0.0, 0.0), new Complex(9.8456, 26.525), new Complex(11.909, 49.181)}};");
                stw.WriteLine("  static public double eta0 = MathNet.Numerics.Trig.Acosh("+ _delta/2.0 + "/"+Radius+"+1);");
                stw.WriteLine("  static public double c = " + Radius + " * Math.Abs(MathNet.Numerics.Trig.Sinh(MathNet.Numerics.Trig.Acosh(" + _delta/2.0 + "/ "+Radius+" + 1)));");
                stw.WriteLine("  static public int mode = " + mode + ";");
                stw.WriteLine("  static public bool even = bool.Parse(\"" + even.ToString() + "\");");
                stw.WriteLine("  static public bool symmetric = bool.Parse(\"" + symmetric.ToString() + "\");");
                stw.WriteLine("  static public ilPSP.Vector VelVec(ilPSP.Vector X) {");
                stw.WriteLine("    Func<double[], double> xi = XX => Math.PI + MathNet.Numerics.Trig.Atan(XX[0] / (XX[1]+c)) - MathNet.Numerics.Trig.Atan(XX[0] / (XX[1]-c));");
                stw.WriteLine("    Func<double[], double> eta = XX => 0.5*Math.Log((Math.Pow(XX[1] + c, 2)+Math.Pow(XX[0], 2)) / (Math.Pow(XX[1] - c, 2)+Math.Pow(XX[0], 2)));");
                stw.WriteLine("    double Psi0 = Streamfunction(xi(X),eta(X));");
                stw.WriteLine("    double eps = 1*BLAS.MachineEps.Sqrt();");
                stw.WriteLine("    double[] X1 = new double[]{X[0]-eps/2, X[1]};");
                stw.WriteLine("    double[] Y1 = new double[]{X[0], X[1]-eps/2};");
                stw.WriteLine("    double[] X2 = new double[]{X[0]+eps/2, X[1]};");
                stw.WriteLine("    double[] Y2 = new double[]{X[0], X[1]+eps/2};");
                stw.WriteLine("    double PsiX1 = Streamfunction(xi(X1),eta(X1));");
                stw.WriteLine("    double PsiX2 = Streamfunction(xi(X2),eta(X2));");
                stw.WriteLine("    double PsiY1 = Streamfunction(xi(Y1),eta(Y1));");
                stw.WriteLine("    double PsiY2 = Streamfunction(xi(Y2),eta(Y2));");
                stw.WriteLine("    double Ux = (PsiY2-PsiY1)/eps;");
                stw.WriteLine("    double Uy = -(PsiX2-PsiX1)/eps;");
                stw.WriteLine("    return new double[] {Ux, Uy};");
                stw.WriteLine("  }");
                stw.WriteLine(" ");
                stw.WriteLine("  static public double VelX(double[] X) {");
                stw.WriteLine("    return VelVec(X).x;");
                stw.WriteLine("  }");
                stw.WriteLine("  static public double VelY(double[] X) {");
                stw.WriteLine("    return VelVec(X).y;");
                stw.WriteLine("  }");
                stw.WriteLine("}");

                return stw.ToString();
            }
        }

        static public Formula Get_VelX(int mode, bool even, bool symmetric, double _delta, double Radius) {
            return new Formula("BoundaryValues.VelX", AdditionalPrefixCode: GetPrefixCode(mode, even, symmetric, _delta, Radius));
        }

        static public Formula Get_VelY(int mode, bool even, bool symmetric, double _delta, double Radius) {
            return new Formula("BoundaryValues.VelY", AdditionalPrefixCode: GetPrefixCode(mode, even, symmetric, _delta, Radius));
        }
    }

In [10]:
Formula Vx = BoundaryValueFactory4Eddies.Get_VelX(1, false, false, 0.02, 1.0);
Vx.Evaluate(new double[] {-0.25,0.02}, 0).Display();
Formula Vy = BoundaryValueFactory4Eddies.Get_VelX(1, true, false, 0.02, 1.0);
Vy.Evaluate(new double[] {-0.25,0.02}, 0).Display();

In [11]:
double[] deltaS = new double[] {
    0.02 // millimeters
    //0.0005, 
    //0.0001, 
    //0.00005, 
    //0.00001, 
    //0.000005, 
    //0.000001 
    }; // micro-meter, 2h when compared to Erbach!
double[] V = new double[] {
  0.0
  //0.1, 
  //0.5, 
  //1.0,
  //5.0, 
  //10.0
    }; // Wall velocities in m/s
bool[] Even = new bool[] {false, true};
bool[] Symmetric = new bool[] {false, true};
int[] Mode = new int[] {1, 2}; // zero-th mode is trivial
double[] xRange = new double[] {0.05, 0.1, 0.15, 0.2, 0.25};

// always same
int Res = 10;
int DGdegree = 5;
double R = 1.0;

// set grid to be saved in database
GridFactory.myDb = myDb;

In [12]:
var controls = new List<XNSE_Control>();

foreach(double delta in deltaS) {
foreach(double V_wall in V) {
foreach(bool symmetric in Symmetric) {
foreach(bool even in Even) {
foreach(int mode in Mode) {
foreach(double range in xRange) {
         var C = new XNSE_Control();
         C.DbPath = myDb.Path;

         C.SetDGdegree(DGdegree);
         C.SetGrid(GridFactory.GenerateGrid4Eddies(Res, delta, R, range));
         C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", Res));
         C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("delta", delta));
         C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Radius", R));

         C.AddBoundaryValue("wall_walze", "VelocityX", BoundaryValueFactory.Get_VelX(delta, V_wall, R));
         C.AddBoundaryValue("wall_walze", "VelocityY", BoundaryValueFactory.Get_VelY(delta, V_wall, R));
         C.AddBoundaryValue("wall_substrat", "VelocityX", BoundaryValueFactory.Get_VelX(delta, V_wall, R));
         C.AddBoundaryValue("wall_substrat", "VelocityY", BoundaryValueFactory.Get_VelY(delta, V_wall, R));
         C.AddBoundaryValue("velocity_inlet", "VelocityX", BoundaryValueFactory4Eddies.Get_VelX(mode, even, symmetric, delta, R));
         C.AddBoundaryValue("velocity_inlet", "VelocityY", BoundaryValueFactory4Eddies.Get_VelY(mode, even, symmetric, delta, R));
       
         //C.AddInitialValue("VelocityX", BoundaryValueFactory4Eddies.Get_VelX(delta, R));
         //C.AddInitialValue("VelocityY", BoundaryValueFactory4Eddies.Get_VelY(delta, R));


         C.TimesteppingMode = AppControl._TimesteppingMode.Steady;

         C.PhysicalParameters.rho_A             = 1.0; // not really relevant as this is steady-state
         C.PhysicalParameters.mu_A              = 1.0;
         C.PhysicalParameters.IncludeConvection = false;
         C.Timestepper_LevelSetHandling         = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;
         
         C.SkipSolveAndEvaluateResidual = false;
       
         C.SessionName = "J" + (20*Res*Res) + "/hR" + (delta/(2.0*R)) + "/" + (even ? "Even" : "Odd") + (symmetric ? "Symmetric" : "Asymmetric") + "_alpha" + mode + "/X"+ range;     
         controls.Add(C);
}
}
}
}
}
}

In [ ]:
controls.ForEach(c => c.SessionName.Display());
controls.Count().Display();
int ctrlCount = controls.Count();

In [14]:
//var C = controls.ElementAt(5);
//C.ImmediatePlotPeriod = 1;
//C.SuperSampling = 2;
//C.savetodb = false;
//C.Run();

### Run Simulations

Workaround so we need to deploy the rather large executables only once!

In [ ]:
//BoSSSshell.WorkflowMgm.ResetProject();

In [16]:
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfMPIProcs = 1);
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [17]:
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(18000);

Assert that, all sessions are present and all finished successful

In [22]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

They are only exported and displayed in the thesis.

In [ ]:
//sessions.ForEach(session=> session.Timesteps.Last().Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());


Only check that all simulations finished successful

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}